## Builder Order Data Comparison

This notebook compares a fresh order export from a customer's PO portal against 
my internal master data file, flagging any jobs where the expected ship date or 
PO number has drifted out of sync between the two.

### Setup
Import libraries and define the file paths for the downloads folder and master 
data file.

In [1]:
# all customer data in this file has been randomly generated so no real customer data is included
import pandas as pd
from pathlib import Path
import xlwings as xw 
import glob
import os

base_dir = Path.home()
download_dir = (
    base_dir
    / "OneDrive"
    / "Downloads"
)
brain_path = (
    base_dir
    / "OneDrive"
    / "Documents"
    / "Master Data.xlsx"
)

### Load the latest order export
I am able to export an updated order list from my customers PO system.  Each time I download one, this grabs the most recently modified file matching the expected naming pattern from my download folder.

In [2]:
pattern = os.path.join(download_dir, "download*.xls")
files = glob.glob(pattern)
if not files:
    raise FileNotFoundError("No future orders file found!")
df = pd.read_excel(max(files, key=os.path.getmtime),skiprows = 6)


### Filter and clean
Keep only rows for the target builder account, and trim down to the columns 
needed for comparison.

In [3]:
# Filter for the target builder account
df = df[df['Account'] == 'Imaginary Homebuilders Inc   Imagine that']
# Keeping only the columns relevant to our comparison
df = df[['Job',"Builder's Order #",'Subdivision','Task Start Date','Total']]
# rename the PO# field
df = df.rename(columns ={"Builder's Order #":'PO#'})

### Extract the street address
The Job field bundles an internal order number together with the address, 
repeated twice. This function isolates just the address for a clean comparison.

In [4]:
def extract_address(job_code):
    """
    This function extracts the street address from the Job field.
    The field contains some irrelevant order number info, followed by a
    '-', followed by the street address repeated twice with a space in
    between (e.g. "12345-123 Main St 123 Main St").
    
    This function counts how many characters exist after the '-', subtracts
    1 to account for the space, then divides by two to get the number of
    characters in the address alone. It then takes that many characters
    from the end of the Job field, leaving something like "123 Main St".
    """
    
    if '-' not in job_code:
        return job_code
    after_dash = job_code.rsplit('-', 1)[-1]
    return after_dash[-((len(after_dash) - 1) // 2):]

df['Job'] = df['Job'].apply(extract_address)


### Handle duplicate entries per address
Each address can appear multiple times in the export, either as an early placeholder 
'order merchandise' note with no PO or dollar amount, or with a PO attached.  Sorting
by dollar amount before deduping keeps the row with the most money visible when multiple
rows exist, but still keeps a row for every Job even if that PO is not yet attached.

In [5]:
# Sort by Job and Total descending, keeping rows with active dollar amounts first
df.sort_values(['Job','Total'], ascending = [True,False], inplace = True)
# this keeps one row per address, always keeping the one with the highest Total
good_rows = df[~df['Job'].duplicated(keep = 'first')].reset_index(drop = True)
# re-sorting the list by date needed after the deduplication
good_rows.sort_values(['Task Start Date'], inplace = True)

### Load the master data file
Tries to read the file directly from Documents. If it's already open, this 
results in a PermissionError, which triggers a fallback to pull the data 
straight from the open Excel session instead.

In [6]:
try:
    brain = pd.read_excel(brain_path)
except PermissionError:
    wb = xw.Book(brain_path)
    sht = wb.sheets["All HP's"]
    full_range = sht.used_range
    data_range = sht.range((1,1), (full_range.last_cell.row, full_range.last_cell.column))
    brain = data_range.options(pd.DataFrame, index=False).value

# making sure all dates are formatted correctly
brain['Date'] = pd.to_datetime(brain['Date'], errors = 'coerce')

### Compare against master data
Map each job's expected date and PO number from the master data file onto the 
current export, so they can quickly be compared side by side.

In [7]:
# create a dictionary with job, date and PO# from master data file
datemap = dict(zip(brain['Job'], brain['Date']))
pomap = dict(zip(brain['Job'], brain['PO#']))

# adds the date and PO# from new dictionaries to the good_rows dataframe
good_rows['brain_date'] = good_rows['Job'].map(datemap)
good_rows['brain_po'] = good_rows['Job'].map(pomap)

### Report mismatches
Print two tables: jobs where the expected date doesn't match, and jobs where 
the PO number doesn't match (including any addresses missing from the master 
data entirely).

In [8]:
# re-ordering columns before our final display
good_rows = good_rows[['Job','Task Start Date','brain_date','PO#','brain_po','Subdivision','Total']]
# This first print will list any address with mis-matched dates, or that are not in the master data file
# showing the address followed by the dates from both sources for quick comparison
good_rows = good_rows.sort_values('Task Start Date')
print('These rows have Date mismatches')
display(good_rows[good_rows['Task Start Date'] != good_rows['brain_date']])

# this re-sorts the list so in the next display, it will start with the PO numbers from both sources
# again for quick comparison.
good_rows = good_rows[['Job','PO#','brain_po','Task Start Date','brain_date','Subdivision','Total']]

print('These rows have PO mismatches')
display(good_rows[good_rows['brain_po'] != good_rows['PO#']])

These rows have Date mismatches


,Job,Task Start Date,brain_date,PO#,brain_po,Subdivision,Total
9,79402 Peterson Drives Apt. 511,2026-08-28,2026-09-02,2MNO5554/012,2MNO5554/012,Highland Meadows,2141.109225
3,33890 Jennifer Squares,2026-09-16,NaT,NaN,NaN,Whispering Pines,0.000000
5,525 Clark Grove Apt. 928,2026-10-05,2026-10-14,1ABC8832/071,1ABC8832/071,Saddleback Village,1777.096950
7,76483 Cameron Trail,2026-10-22,NaT,1LKS4111/026,NaN,Cimarron Hills,2713.859975
2,3287 Katelyn Wall Apt. 226,2026-10-27,NaT,2MNO5677/035,NaN,Verde Vista,1046.268725
4,43321 Brittany Bypass,2026-12-13,2026-12-06,1LKS4152/089,1LKS4152/089,Oak Creek Ridge,1723.783825


These rows have PO mismatches


,Job,PO#,brain_po,Task Start Date,brain_date,Subdivision,Total
3,33890 Jennifer Squares,NaN,NaN,2026-09-16,NaT,Whispering Pines,0.000000
8,76724 John Points Suite 969,NaN,NaN,2026-10-15,2026-10-15,Timberline Acres,0.000000
6,55940 Herrera Lane Apt. 849,1LKS4198/044,1LKS4198/043,2026-10-20,2026-10-20,Copper Canyon,2841.443425
0,05641 Robin Port,NaN,NaN,2026-10-21,2026-10-21,Lone Star Crossing,0.000000
7,76483 Cameron Trail,1LKS4111/026,NaN,2026-10-22,NaT,Cimarron Hills,2713.859975
2,3287 Katelyn Wall Apt. 226,2MNO5677/035,NaN,2026-10-27,NaT,Verde Vista,1046.268725
